In [ ]:
# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/gdrive')

print("✓ Google Drive mounted")

In [ ]:
# ============================================================
# CELL 2 — INSTALL DEPENDENCIES
# ============================================================

!apt-get update -qq && apt-get install -y ffmpeg build-essential
!pip install -q librosa soundfile numpy scipy tqdm pyworld

print("✓ Dependencies installed")

In [ ]:
# ============================================================
# CELL 3 — LANGUAGE DATASET CONFIGURATION
# ============================================================

import os

LANGUAGES = {

    'hindi': {
        'input': '/content/drive/MyDrive/five_songs_dataset/hindi',
        'output': '/content/drive/MyDrive/five_language_output/hindi/rvc'
    },

    'bengali': {
        'input': '/content/drive/MyDrive/five_songs_dataset/bengali',
        'output': '/content/drive/MyDrive/five_language_output/bengali/rvc'
    }
}

# Create the required output and reference directories.
for lang, config in LANGUAGES.items():

    os.makedirs(
        config['output'],
        exist_ok=True
    )

    os.makedirs(
        f"{config['output']}/references",
        exist_ok=True
    )

print("✓ Hindi and Bengali output directories created")

In [ ]:
# ============================================================
# CELL 4 — LOAD AUDIO FILES
# ============================================================

def load_audio_files(path):

    audio_files = []

    for root, dirs, files in os.walk(path):

        for file in files:

            # Accept the same audio formats as the original
            # RVC pipeline.
            if file.lower().endswith(
                ('.wav', '.mp3', '.flac', '.m4a')
            ):

                audio_files.append(
                    os.path.join(root, file)
                )

    return sorted(audio_files)


dataset_info = {}

for lang, config in LANGUAGES.items():

    files = load_audio_files(
        config['input']
    )

    dataset_info[lang] = files

    print(
        f"{lang.capitalize()}: "
        f"{len(files)} songs found"
    )


total_songs = sum(
    len(files)
    for files in dataset_info.values()
)

print(
    f"\n✓ Total songs across Hindi and Bengali: "
    f"{total_songs}"
)

In [ ]:
# ============================================================
# CELL 5 — CREATE RVC REFERENCE SAMPLES
# ============================================================

import librosa
import soundfile as sf


def create_reference_sample(
    audio_path,
    ref_path,
    duration=30
):
    """
    Create a language-specific reference sample.

    The original RVC pipeline loads the reference at 22.05 kHz
    and uses a 30-second sample for pitch estimation.
    """

    audio, sr = librosa.load(
        audio_path,
        sr=22050,
        duration=duration
    )

    # Normalize the reference before using it.
    sample = librosa.util.normalize(
        audio
    )

    sf.write(
        ref_path,
        sample,
        sr
    )


reference_samples = {}


for lang, files in dataset_info.items():

    if not files:
        continue

    ref_path = (
        f"{LANGUAGES[lang]['output']}/"
        f"references/{lang}_rvc_ref.wav"
    )

    create_reference_sample(
        files[0],
        ref_path
    )

    reference_samples[lang] = ref_path

    print(
        f"{lang.capitalize()}: "
        f"Reference created from "
        f"{os.path.basename(files[0])}"
    )


print("\n✓ Reference samples for RVC created")

In [ ]:
# ============================================================
# CELL 6 — RVC-BASED VOICE CONVERSION
# ============================================================

import numpy as np
import pyworld as pw


def rvc_convert(
    source,
    reference,
    sr
):
    """
    Perform reference-guided singing voice conversion.

    WORLD is used to estimate:
        - Fundamental frequency (F0)
        - Spectral envelope
        - Aperiodicity

    The reference recording determines the pitch ratio,
    while the source spectral representation is synthesized
    using the converted F0.
    """

    # WORLD requires floating-point audio.
    source = source.astype(
        np.float64
    )

    reference = reference.astype(
        np.float64
    )

    # --------------------------------------------------------
    # SOURCE ANALYSIS
    # --------------------------------------------------------

    f0_src, t_src = pw.dio(
        source,
        sr
    )

    f0_src = pw.stonemask(
        source,
        f0_src,
        t_src,
        sr
    )

    sp_src = pw.cheaptrick(
        source,
        f0_src,
        t_src,
        sr
    )

    ap_src = pw.d4c(
        source,
        f0_src,
        t_src,
        sr
    )

    # --------------------------------------------------------
    # REFERENCE ANALYSIS
    # --------------------------------------------------------

    f0_ref, t_ref = pw.dio(
        reference,
        sr
    )

    f0_ref = pw.stonemask(
        reference,
        f0_ref,
        t_ref,
        sr
    )

    # Keep only voiced F0 values when calculating the
    # reference-to-source pitch ratio.
    src_voiced = f0_src[
        f0_src > 0
    ]

    ref_voiced = f0_ref[
        f0_ref > 0
    ]

    # If a valid source F0 is unavailable, retain the
    # original pitch rather than applying an invalid ratio.
    if len(src_voiced) > 0 and len(ref_voiced) > 0:

        pitch_ratio = (
            np.mean(ref_voiced)
            /
            np.mean(src_voiced)
        )

    else:

        pitch_ratio = 1.0

    # --------------------------------------------------------
    # PITCH CONVERSION
    # --------------------------------------------------------

    f0_converted = np.clip(
        f0_src * pitch_ratio,
        50,
        800
    )

    # --------------------------------------------------------
    # AUDIO SYNTHESIS
    # --------------------------------------------------------

    converted = pw.synthesize(
        f0_converted,
        sp_src,
        ap_src,
        sr
    )

    # Normalize generated audio.
    converted = librosa.util.normalize(
        converted
    )

    # Remove low-level silence from the output.
    converted, _ = librosa.effects.trim(
        converted,
        top_db=20
    )

    return converted

In [ ]:
# ============================================================
# CELL 7 — GENERATE RVC SYNTHETIC SINGING VOICE SAMPLES
# ============================================================

from tqdm import tqdm
import gc


print("=" * 70)
print("STEP 2: RVC-BASED VOICE CONVERSION")
print("=" * 70)


conversion_results = {}


for lang, config in LANGUAGES.items():

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"🎵 PROCESSING {lang.upper()} SONGS"
    )

    print(
        f"{'=' * 70}"
    )

    files = dataset_info[lang]

    if not files:

        print(
            f"No files found for {lang}, skipping."
        )

        continue

    conversion_results[lang] = []

    # Use the first recording as the language-specific
    # reference, matching the original notebook.
    reference_path = reference_samples[lang]

    reference, sr = librosa.load(
        reference_path,
        sr=22050
    )

    # Start from files[1:] because files[0] is used
    # as the reference recording.
    for i, audio_path in enumerate(
        tqdm(
            files[1:],
            desc=f"RVC {lang}"
        )
    ):

        out_file = (
            f"rvc_{lang}_{i + 1:03d}.wav"
        )

        out_path = os.path.join(
            config['output'],
            out_file
        )

        try:

            # Process each source recording at 22.05 kHz
            # and limit processing to 30 seconds, matching
            # the original notebook.
            source, _ = librosa.load(
                audio_path,
                sr=22050,
                duration=30
            )

            converted_audio = rvc_convert(
                source,
                reference,
                sr
            )

            # Save generated audio as 16-bit PCM WAV.
            sf.write(
                out_path,
                converted_audio,
                sr,
                subtype='PCM_16'
            )

            conversion_results[lang].append(
                out_file
            )

            print(
                f"✓ Saved {out_file}"
            )

        except Exception as e:

            print(
                f"Error on {out_file}: {e}"
            )

        # Free memory after every recording so that
        # Colab can process larger language subsets.
        if 'source' in locals():
            del source

        if 'converted_audio' in locals():
            del converted_audio

        gc.collect()

    del reference
    gc.collect()


print(
    "\n✓ RVC PROCESSING COMPLETED"
)

In [ ]:
# ============================================================
# CELL 8 — FINAL CONVERSION SUMMARY
# ============================================================

print("=" * 70)
print("🎉 RVC CONVERSION SUMMARY")
print("=" * 70)

total_converted = 0

for lang, results in conversion_results.items():

    count = len(results)

    total_converted += count

    print(
        f"{lang.capitalize():10s}: "
        f"{count} songs converted"
    )

print("-" * 70)

print(
    f"TOTAL: {total_converted} songs converted "
    f"across {len(conversion_results)} languages!"
)

print("=" * 70)